#  Titanic Dataset: Complete Data Cleaning & Transformation Pipeline

##  Project Overview
This notebook processes and prepares the **Titanic** passenger dataset using Data Cleaning, Structural Transformation, and Feature Engineering techniques to get the data ready for Exploratory Data Analysis (EDA) and Machine Learning models.

---

##  Summary of Pipeline Steps

### 1. Data Cleaning & Missing Values Handling
* **`Age`**: Imputed missing values using the **Median** to prevent skewness from outliers.
* **`Embarked`**: Imputed missing values using the **Mode** (most frequent value).
* **`Cabin`**: Replaced missing values with an **`"Unknown"`** category label.

### 2. Data Type Conversions
* **`Fare`**: Casted data type from `float` to `Integer` after processing to remove unnecessary decimals.

### 3. Feature Engineering
* **`Title`**: Extracted passenger titles (`Mr`, `Mrs`, `Miss`, etc.) from the `Name` column using Regular Expressions (`Regex`).
* **`FamilySize`**: Calculated total family members aboard by summing (`SibSp + Parch + 1`).
* **`Fare_Category`**: Binned ticket fares into three categories (`Low`, `Medium`, `High`) based on price ranges.

### 4. Advanced Aggregation & Joins
* **Aggregation**: Calculated `max`, `min`, and `mean`/`avg` fare prices grouped by port of embarkation (`Embarked`).
* **Merging / Joins**: Joined external description tables with the main dataset to map passenger classes (`Pclass`) to their detailed descriptions.

---

## Summary of Tech Stack & Methods Used

| Pipeline Step | Pandas Method | PySpark Equivalent |
| :--- | :--- | :--- |
| **Handling Missing Data** | `.fillna()` / `.median()` / `.mode()` | `df.fillna()` / `approxQuantile()` |
| **Casting Types** | `.astype(int)` | `col().cast("integer")` |
| **String Extraction** | `.str.extract()` | `regexp_extract()` |
| **Conditional Logic** | `pd.cut()` / `np.select()` | `when().otherwise()` |
| **Aggregation** | `.groupby().agg()` | `.groupBy().agg()` |
| **Data Merging** | `pd.merge()` | `.join()` |

---

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [4]:
import urllib.request

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
urllib.request.urlretrieve(url, "titanic.csv")


('titanic.csv', <http.client.HTTPMessage at 0x7b44c832a8d0>)

In [5]:
df = spark.read.csv("titanic.csv", header=True, inferSchema=True)
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [6]:
pclass1_over30=df.filter((df['Age']>30)&(df['Pclass']==1) )
pclass1_over30.show()

+-----------+--------+------+--------------------+------+----+-----+-----+-----------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|     Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+-----------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|   PC 17599|71.2833|  C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|     113803|   53.1| C123|       S|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|      17463|51.8625|  E46|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|     113783|  26.55| C103|       S|
|         31|       0|     1|Uruchurtu, Don. M...|  male|40.0|    0|    0|   PC 17601|27.7208| NULL|       C|
|         36|       0|     1|Holverson, Mr. Al...|  male|42.0|    1|    0|     113789|   52.0| NULL|       S|
|         

In [7]:
survived=df.filter((df['Survived']==1)&(df['Embarked']=="C"))
survived.show()

+-----------+--------+------+--------------------+------+----+-----+-----+-------------+--------+-------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|       Ticket|    Fare|  Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+-------------+--------+-------+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|     PC 17599| 71.2833|    C85|       C|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|       237736| 30.0708|   NULL|       C|
|         20|       1|     3|Masselmani, Mrs. ...|female|NULL|    0|    0|         2649|   7.225|   NULL|       C|
|         32|       1|     1|Spencer, Mrs. Wil...|female|NULL|    1|    0|     PC 17569|146.5208|    B78|       C|
|         37|       1|     3|    Mamee, Mr. Hanna|  male|NULL|    0|    0|         2677|  7.2292|   NULL|       C|
|         40|       1|     3|Nicola-Yarred, Mi...|female|14.0|    1|    0|      

In [35]:
df=df.withColumn("avg_group" , when(col("Age")<18 ,"Child").otherwise("Adult") )
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+---------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|avg_group|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+---------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|    Adult|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|    Adult|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|    Adult|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|    Adult|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|    Adult|


In [14]:
avg_age=df.select(avg("Age")).first()[0]


In [25]:
avg_age_arounded=round(avg_age,1)
df=df.fillna({'Age' : avg_age_arounded})
df.select("Name", "Age").show(10)

+--------------------+----+
|                Name| Age|
+--------------------+----+
|Braund, Mr. Owen ...|22.0|
|Cumings, Mrs. Joh...|38.0|
|Heikkinen, Miss. ...|26.0|
|Futrelle, Mrs. Ja...|35.0|
|Allen, Mr. Willia...|35.0|
|    Moran, Mr. James|29.7|
|McCarthy, Mr. Tim...|54.0|
|Palsson, Master. ...| 2.0|
|Johnson, Mrs. Osc...|27.0|
|Nasser, Mrs. Nich...|14.0|
+--------------------+----+
only showing top 10 rows



In [32]:
ordered=df.orderBy(col("Fare").desc())
ordered.show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+---------------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|          Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+---------------+--------+
|        259|       1|     1|    Ward, Miss. Anna|female|35.0|    0|    0|PC 17755|512.3292|           NULL|       C|
|        680|       1|     1|Cardeza, Mr. Thom...|  male|36.0|    0|    1|PC 17755|512.3292|    B51 B53 B55|       C|
|        738|       1|     1|Lesurer, Mr. Gust...|  male|35.0|    0|    0|PC 17755|512.3292|           B101|       C|
|        342|       1|     1|Fortune, Miss. Al...|female|24.0|    3|    2|   19950|   263.0|    C23 C25 C27|       S|
|         89|       1|     1|Fortune, Miss. Ma...|female|23.0|    3|    2|   19950|   263.0|    C23 C25 C27|       S|
|        439|       0|     1|   Fortune, Mr. Mark|  male

In [40]:
df_grouped = df.groupBy("avg_group").agg(
    count("*").alias("Total_Passengers"),
    round(avg("Fare"), 2).alias("Avg_Fare"),
    sum("Survived").alias("Total_Survived")
)

df_grouped.show()

+---------+----------------+--------+--------------+
|avg_group|Total_Passengers|Avg_Fare|Total_Survived|
+---------+----------------+--------+--------------+
|    Adult|             778|   32.35|           281|
|    Child|             113|   31.22|            61|
+---------+----------------+--------+--------------+



In [ ]:
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+---------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|avg_group|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+---------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|    Adult|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|    Adult|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|    Adult|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|    Adult|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|    Adult|


In [45]:
df_group2=df.groupBy('Pclass').agg(max('Fare').alias('max_fare') , min('Fare').alias("min_fare"),round(avg('Age'),1).alias("avg_age")).orderBy(col("max_fare").desc())
df_group2.show()

+------+--------+--------+-------+
|Pclass|max_fare|min_fare|avg_age|
+------+--------+--------+-------+
|     1|512.3292|     0.0|   37.0|
|     2|    73.5|     0.0|   29.9|
|     3|   69.55|     0.0|   26.4|
+------+--------+--------+-------+



In [48]:
df_filtering=df.filter((col('Embarked')=='C')&(col('Fare')>100))
df_filtering.select("Name","Pclass","Fare","Embarked").show()

+--------------------+------+--------+--------+
|                Name|Pclass|    Fare|Embarked|
+--------------------+------+--------+--------+
|Spencer, Mrs. Wil...|     1|146.5208|       C|
|Baxter, Mr. Quigg...|     1|247.5208|       C|
|Lurette, Miss. Elise|     1|146.5208|       C|
|Newell, Miss. Mad...|     1| 113.275|       C|
|    Ward, Miss. Anna|     1|512.3292|       C|
|Baxter, Mrs. Jame...|     1|247.5208|       C|
|Fleming, Miss. Ma...|     1|110.8833|       C|
|Penasco y Castell...|     1|   108.9|       C|
|Ryerson, Miss. Em...|     1| 262.375|       C|
|Spedden, Mrs. Fre...|     1|   134.5|       C|
|Young, Miss. Mari...|     1|135.6333|       C|
|Burns, Miss. Eliz...|     1|   134.5|       C|
| Ringhini, Mr. Sante|     1|135.6333|       C|
|Widener, Mr. Harr...|     1|   211.5|       C|
|Bidois, Miss. Ros...|     1| 227.525|       C|
|Newell, Miss. Mar...|     1| 113.275|       C|
|Penasco y Castell...|     1|   108.9|       C|
| LeRoy, Miss. Bertha|     1| 106.425|  

In [49]:
ports_data = [("C", "Cherbourg"), ("Q", "Queenstown"), ("S", "Southampton")]
ports_df = spark.createDataFrame(ports_data, ["Embarked_Code", "Port_Name"])

In [50]:
ports_df.show()

+-------------+-----------+
|Embarked_Code|  Port_Name|
+-------------+-----------+
|            C|  Cherbourg|
|            Q| Queenstown|
|            S|Southampton|
+-------------+-----------+



In [ ]:
df_joined = df.join(
    ports_df, 
    df["Embarked"] == ports_df["Embarked_Code"], 
    how="left"
)

In [53]:
df_joined.select("PassengerId", "Name", "Embarked", "Port_Name").show(10, truncate=False)

+-----------+---------------------------------------------------+--------+-----------+
|PassengerId|Name                                               |Embarked|Port_Name  |
+-----------+---------------------------------------------------+--------+-----------+
|6          |Moran, Mr. James                                   |Q       |Queenstown |
|2          |Cumings, Mrs. John Bradley (Florence Briggs Thayer)|C       |Cherbourg  |
|10         |Nasser, Mrs. Nicholas (Adele Achem)                |C       |Cherbourg  |
|1          |Braund, Mr. Owen Harris                            |S       |Southampton|
|3          |Heikkinen, Miss. Laina                             |S       |Southampton|
|4          |Futrelle, Mrs. Jacques Heath (Lily May Peel)       |S       |Southampton|
|5          |Allen, Mr. William Henry                           |S       |Southampton|
|7          |McCarthy, Mr. Timothy J                            |S       |Southampton|
|8          |Palsson, Master. Gosta Leonard

In [56]:
classes_data=[
    ('1','First Class - Luxury'),
    ('2','Second Class - Mid Range'),
    ('3','Third Class - Economy')
]
df_classed=spark.createDataFrame(classes_data,["Class_ID", "Class_Description"])
df_join_classes=df.join(df_classed , df['Pclass']==df_classed['Class_ID'], how="left")
df_join_classes.select('PassengerId', 'Name', 'Pclass', 'Class_Description').show()

+-----------+--------------------+------+--------------------+
|PassengerId|                Name|Pclass|   Class_Description|
+-----------+--------------------+------+--------------------+
|          2|Cumings, Mrs. Joh...|     1|First Class - Luxury|
|          4|Futrelle, Mrs. Ja...|     1|First Class - Luxury|
|          7|McCarthy, Mr. Tim...|     1|First Class - Luxury|
|         12|Bonnell, Miss. El...|     1|First Class - Luxury|
|          1|Braund, Mr. Owen ...|     3|Third Class - Eco...|
|          3|Heikkinen, Miss. ...|     3|Third Class - Eco...|
|          5|Allen, Mr. Willia...|     3|Third Class - Eco...|
|          6|    Moran, Mr. James|     3|Third Class - Eco...|
|          8|Palsson, Master. ...|     3|Third Class - Eco...|
|          9|Johnson, Mrs. Osc...|     3|Third Class - Eco...|
|         11|Sandstrom, Miss. ...|     3|Third Class - Eco...|
|         13|Saundercock, Mr. ...|     3|Third Class - Eco...|
|         14|Andersson, Mr. An...|     3|Third Class - 

In [58]:
df=df.withColumn("Fare",col("Fare").cast('integer'))

In [61]:
df = df.fillna({"Cabin": "Unknown"})

In [63]:
df=df.withColumn("Name_length", length('Name'))

In [72]:
df_group_fare=df.groupBy('Embarked').agg(
    max('Fare').alias("max_fare"),
    min('Fare').alias("min_fare"),
    avg("Fare").alias("avg_fare")

)
df_group_fare.show()

+--------+--------+--------+------------------+
|Embarked|max_fare|min_fare|          avg_fare|
+--------+--------+--------+------------------+
|       Q|      90|       6|12.675324675324676|
|    NULL|      80|      80|              80.0|
|       C|     512|       4|59.523809523809526|
|       S|     263|       0|26.684782608695652|
+--------+--------+--------+------------------+



In [73]:
df = df.withColumn(
    "Fare_Category",
    when(col("Fare") < 15, "Low")
    .when((col("Fare") >= 15) & (col("Fare") <= 50), "Medium")
    .otherwise("High")
)

In [74]:
median_age = df.stat.approxQuantile("Age", [0.5], 0.0)[0]
df = df.fillna({"Age": median_age})
mode_embarked = df.groupBy("Embarked").count().orderBy(col("count").desc()).first()[0]
df = df.fillna({"Embarked": mode_embarked})
df = df.withColumn("Title", regexp_extract(col("Name"), r" ([A-Za-z]+)\.", 1))